In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install rdkit
!pip install torch_geometric==2.5.3

In [ ]:
import os
import sys
import torch
import time
from tqdm import tqdm
import datetime
doc_name = "/content/drive/MyDrive/HeckLit-Code"
sys.path.append(doc_name)
from utils.rxn import *
from utils.molecule import *
from utils.dataset_analysis import *
from models.DeepLearnModel import *
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

In [ ]:
rs_list = [1,2,3,4,5]
eval_metrics = np.zeros((8*(1+len(rs_list)), 6))
columns = ['train_R2', 'train_RMSE','train_MAE','test_R2','test_RMSE','test_MAE']
models = ["RF-RXNFP", "RF-DRFP", "Xgboost-RXNFP", "Xgboost-DRFP", "SVM-RXNFP", "SVM-DRFP", "kNN-RXNFP", "kNN-DRFP"]
index = []
for model in models:
  for rs in rs_list:
    index.append("%s-%s" % (model, rs))
  index.append("avg±std")
eval_metrics = pd.DataFrame(eval_metrics, columns=columns, index=index)

In [ ]:
for rs in rs_list:
  # 1. import data
  data = pd.read_excel("%s/data/Heck/JCP processed data.xlsx" % doc_name)
  random_state = rs
  data = data.sample(random_state=random_state, frac=1).reset_index(drop=True)

  # 2. build dataset & dataloader
  rxnfp_set = list()
  drfp_set = list()
  yield_set = list()
  len_drfp = 0

  for i in tqdm(range(data.shape[0])):
    # features
    rxnfp = read_rxnfp(data.loc[i]["rxnfp"])
    drfp = read_drfp(data.loc[i]["drfp"])
    len_drfp = drfp.shape[0]
    # label
    y = np.array(float(data.loc[i]["ee"]) / 100)

    rxnfp_set.append(rxnfp)
    drfp_set.append(drfp)
    yield_set.append(y)

  # split of train & test set
  ratio = 0.8

  # rxnfp
  batch = len(rxnfp_set)
  rxnfp_trainset = [rxnfp_set[0: int(ratio * batch)], yield_set[0: int(ratio * batch)]]
  rxnfp_testset = [rxnfp_set[int(ratio * batch) + 1:], yield_set[int(ratio * batch) + 1:]]

  # drfp
  batch = len(drfp_set)
  drfp_trainset = [drfp_set[0: int(ratio * batch)], yield_set[0: int(ratio * batch)]]
  drfp_testset = [drfp_set[int(ratio * batch) + 1:], yield_set[int(ratio * batch) + 1:]]

  # report
  dir_path = "%s/exp/Heck_JCP/ML_rs=%s_%s" % (doc_name, rs, datetime.datetime.now())
  os.mkdir("%s" % dir_path)
  f = open("%s/Model_Training_Report.txt" % dir_path, mode="w")

  # record
  f.write("params:\n")
  f.write("random_state=%s\n" % random_state)
  f.write("ratio=%s\n" % ratio)
  f.write("\n")

  # 3.Machine Learning methods
  # RandomForest
  from sklearn.ensemble import RandomForestRegressor

  # Train
  print("RandomForest Start Training")
  start = time.time()

  rxnfp_rf = RandomForestRegressor(n_estimators=100, random_state=0)
  drfp_rf = RandomForestRegressor(n_estimators=100, random_state=0)
  rxnfp_rf.fit(rxnfp_trainset[0], rxnfp_trainset[1])
  drfp_rf.fit(drfp_trainset[0], drfp_trainset[1])

  print("Finish Training")
  print("Training Time: %.2f s" % (time.time() - start))  # 截止时间

  # Eval
  # trainset
  # R2
  rxnfp_train_R2 = R2(rxnfp_rf.predict(rxnfp_trainset[0]), np.array(rxnfp_trainset[1]))
  drfp_train_R2 = R2(drfp_rf.predict(drfp_trainset[0]), np.array(drfp_trainset[1]))
  # RMSE
  rxnfp_train_RMSE = RMSE(rxnfp_rf.predict(rxnfp_trainset[0]), np.array(rxnfp_trainset[1]))
  drfp_train_RMSE = RMSE(drfp_rf.predict(drfp_trainset[0]), np.array(drfp_trainset[1]))
  # MAE
  rxnfp_train_MAE = MAE(rxnfp_rf.predict(rxnfp_trainset[0]), np.array(rxnfp_trainset[1]))
  drfp_train_MAE = MAE(drfp_rf.predict(drfp_trainset[0]), np.array(drfp_trainset[1]))

  # R2
  rxnfp_test_R2 = R2(rxnfp_rf.predict(rxnfp_testset[0]), np.array(rxnfp_testset[1]))
  drfp_test_R2 = R2(drfp_rf.predict(drfp_testset[0]), np.array(drfp_testset[1]))
  # RMSE
  rxnfp_test_RMSE = RMSE(rxnfp_rf.predict(rxnfp_testset[0]), np.array(rxnfp_testset[1]))
  drfp_test_RMSE = RMSE(drfp_rf.predict(drfp_testset[0]), np.array(drfp_testset[1]))
  # MAE
  rxnfp_test_MAE = MAE(rxnfp_rf.predict(rxnfp_testset[0]), np.array(rxnfp_testset[1]))
  drfp_test_MAE = MAE(drfp_rf.predict(drfp_testset[0]), np.array(drfp_testset[1]))

  # Record
  f.write("RandomForest:\n")
  f.write("params:\n")
  f.write("rxnfp RF:%s\n" % rxnfp_rf.n_estimators)
  f.write("drfp RF:%s\n" % drfp_rf.n_estimators)

  f.write("rxnfp:\n")
  f.write("rxnfp_train_R2=%s\n" % rxnfp_train_R2)
  f.write("rxnfp_train_RMSE=%s\n" % rxnfp_train_RMSE)
  f.write("rxnfp_train_MAE=%s\n" % rxnfp_train_MAE)
  f.write("rxnfp_test_R2=%s\n" % rxnfp_test_R2)
  f.write("rxnfp_test_RMSE=%s\n" % rxnfp_test_RMSE)
  f.write("rxnfp_test_MAE=%s\n" % rxnfp_test_MAE)
  f.write("drfp:\n")
  f.write("drfp_train_R2=%s\n" % drfp_train_R2)
  f.write("drfp_train_RMSE=%s\n" % drfp_train_RMSE)
  f.write("drfp_train_MAE=%s\n" % drfp_train_MAE)
  f.write("drfp_test_R2=%s\n" % drfp_test_R2)
  f.write("drfp_test_RMSE=%s\n" % drfp_test_RMSE)
  f.write("drfp_test_MAE=%s\n" % drfp_test_MAE)
  f.write("\n")

  eval_metrics.loc["RF-RXNFP-%s" % rs]["train_R2"] = rxnfp_train_R2
  eval_metrics.loc["RF-RXNFP-%s" % rs]["train_RMSE"] = rxnfp_train_RMSE
  eval_metrics.loc["RF-RXNFP-%s" % rs]["train_MAE"] = rxnfp_train_MAE
  eval_metrics.loc["RF-RXNFP-%s" % rs]["test_R2"] = rxnfp_test_R2
  eval_metrics.loc["RF-RXNFP-%s" % rs]["test_RMSE"] = rxnfp_test_RMSE
  eval_metrics.loc["RF-RXNFP-%s" % rs]["test_MAE"] = rxnfp_test_MAE
  eval_metrics.loc["RF-DRFP-%s" % rs]["train_R2"] = drfp_train_R2
  eval_metrics.loc["RF-DRFP-%s" % rs]["train_RMSE"] = drfp_train_RMSE
  eval_metrics.loc["RF-DRFP-%s" % rs]["train_MAE"] = drfp_train_MAE
  eval_metrics.loc["RF-DRFP-%s" % rs]["test_R2"] = drfp_test_R2
  eval_metrics.loc["RF-DRFP-%s" % rs]["test_RMSE"] = drfp_test_RMSE
  eval_metrics.loc["RF-DRFP-%s" % rs]["test_MAE"] = drfp_test_MAE

  # RF Figure
  # RXNFP
  fig = plt.figure(dpi=300, figsize=(10, 5))
  # Test set performance
  tr = np.array(rxnfp_testset[1]).flatten() * 100
  pr = rxnfp_rf.predict(rxnfp_testset[0]).flatten() * 100
  plt.scatter(pr, tr, alpha=0.7, marker=".")
  plt.xlabel("Predicted Yield", fontsize=10)
  plt.ylabel("Observed Yield", fontsize=10)
  x = np.linspace(0, 100, 100)
  y = np.linspace(0, 100, 100)
  plt.plot(x, y, linestyle="--", color="r")
  plt.title("Test set performance", fontsize=15)

  # DRFP
  # Figure
  fig = plt.figure(dpi=300, figsize=(10, 5))
  # Test set performance
  tr = np.array(drfp_testset[1]).flatten() * 100
  pr = drfp_rf.predict(drfp_testset[0]).flatten() * 100
  plt.scatter(pr, tr, alpha=0.7, marker=".")
  plt.xlabel("Predicted Yield", fontsize=10)
  plt.ylabel("Observed Yield", fontsize=10)
  x = np.linspace(0, 100, 100)
  y = np.linspace(0, 100, 100)
  plt.plot(x, y, linestyle="--", color="r")
  plt.title("Test set performance", fontsize=15)

  # xgboost
  from xgboost import XGBRegressor

  # Train
  print("XGBoost Start Training")
  start = time.time()

  rxnfp_xgb = XGBRegressor(n_estimators=300)
  drfp_xgb = XGBRegressor(n_estimators=300)
  rxnfp_xgb.fit(rxnfp_trainset[0], rxnfp_trainset[1])
  drfp_xgb.fit(drfp_trainset[0], drfp_trainset[1])

  print("Finish Training")
  print("Training Time: %.2f s" % (time.time() - start))  # 截止时间

  # Eval
  # trainset
  # R2
  rxnfp_train_R2 = R2(rxnfp_xgb.predict(rxnfp_trainset[0]), np.array(rxnfp_trainset[1]))
  drfp_train_R2 = R2(drfp_xgb.predict(drfp_trainset[0]), np.array(drfp_trainset[1]))
  # RMSE
  rxnfp_train_RMSE = RMSE(rxnfp_xgb.predict(rxnfp_trainset[0]), np.array(rxnfp_trainset[1]))
  drfp_train_RMSE = RMSE(drfp_xgb.predict(drfp_trainset[0]), np.array(drfp_trainset[1]))
  # MAE
  rxnfp_train_MAE = MAE(rxnfp_xgb.predict(rxnfp_trainset[0]), np.array(rxnfp_trainset[1]))
  drfp_train_MAE = MAE(drfp_xgb.predict(drfp_trainset[0]), np.array(drfp_trainset[1]))

  # R2
  rxnfp_test_R2 = R2(rxnfp_xgb.predict(rxnfp_testset[0]), np.array(rxnfp_testset[1]))
  drfp_test_R2 = R2(drfp_xgb.predict(drfp_testset[0]), np.array(drfp_testset[1]))
  # RMSE
  rxnfp_test_RMSE = RMSE(rxnfp_xgb.predict(rxnfp_testset[0]), np.array(rxnfp_testset[1]))
  drfp_test_RMSE = RMSE(drfp_xgb.predict(drfp_testset[0]), np.array(drfp_testset[1]))
  # MAE
  rxnfp_test_MAE = MAE(rxnfp_xgb.predict(rxnfp_testset[0]), np.array(rxnfp_testset[1]))
  drfp_test_MAE = MAE(drfp_xgb.predict(drfp_testset[0]), np.array(drfp_testset[1]))

  # Record
  f.write("XGBoost:\n")
  f.write("params:\n")
  f.write("rxnfp XGB:%s\n" % rxnfp_xgb.n_estimators)
  f.write("drfp XGB:%s\n" % drfp_xgb.n_estimators)

  f.write("rxnfp:\n")
  f.write("rxnfp_train_R2=%s\n" % rxnfp_train_R2)
  f.write("rxnfp_train_RMSE=%s\n" % rxnfp_train_RMSE)
  f.write("rxnfp_train_MAE=%s\n" % rxnfp_train_MAE)
  f.write("rxnfp_test_R2=%s\n" % rxnfp_test_R2)
  f.write("rxnfp_test_RMSE=%s\n" % rxnfp_test_RMSE)
  f.write("rxnfp_test_MAE=%s\n" % rxnfp_test_MAE)
  f.write("drfp:\n")
  f.write("drfp_train_R2=%s\n" % drfp_train_R2)
  f.write("drfp_train_RMSE=%s\n" % drfp_train_RMSE)
  f.write("drfp_train_MAE=%s\n" % drfp_train_MAE)
  f.write("drfp_test_R2=%s\n" % drfp_test_R2)
  f.write("drfp_test_RMSE=%s\n" % drfp_test_RMSE)
  f.write("drfp_test_MAE=%s\n" % drfp_test_MAE)
  f.write("\n")

  eval_metrics.loc["Xgboost-RXNFP-%s" % rs]["train_R2"] = rxnfp_train_R2
  eval_metrics.loc["Xgboost-RXNFP-%s" % rs]["train_RMSE"] = rxnfp_train_RMSE
  eval_metrics.loc["Xgboost-RXNFP-%s" % rs]["train_MAE"] = rxnfp_train_MAE
  eval_metrics.loc["Xgboost-RXNFP-%s" % rs]["test_R2"] = rxnfp_test_R2
  eval_metrics.loc["Xgboost-RXNFP-%s" % rs]["test_RMSE"] = rxnfp_test_RMSE
  eval_metrics.loc["Xgboost-RXNFP-%s" % rs]["test_MAE"] = rxnfp_test_MAE
  eval_metrics.loc["Xgboost-DRFP-%s" % rs]["train_R2"] = drfp_train_R2
  eval_metrics.loc["Xgboost-DRFP-%s" % rs]["train_RMSE"] = drfp_train_RMSE
  eval_metrics.loc["Xgboost-DRFP-%s" % rs]["train_MAE"] = drfp_train_MAE
  eval_metrics.loc["Xgboost-DRFP-%s" % rs]["test_R2"] = drfp_test_R2
  eval_metrics.loc["Xgboost-DRFP-%s" % rs]["test_RMSE"] = drfp_test_RMSE
  eval_metrics.loc["Xgboost-DRFP-%s" % rs]["test_MAE"] = drfp_test_MAE

  # XGB Figure
  # RXNFP
  fig = plt.figure(dpi=300, figsize=(10, 5))
  # Test set performance
  tr = np.array(rxnfp_testset[1]).flatten() * 100
  pr = rxnfp_xgb.predict(rxnfp_testset[0]).flatten() * 100
  plt.scatter(pr, tr, alpha=0.7, marker=".")
  plt.xlabel("Predicted Yield", fontsize=10)
  plt.ylabel("Observed Yield", fontsize=10)
  x = np.linspace(0, 100, 100)
  y = np.linspace(0, 100, 100)
  plt.plot(x, y, linestyle="--", color="r")
  plt.title("Test set performance", fontsize=15)

  # DRFP
  # Figure
  fig = plt.figure(dpi=300, figsize=(10, 5))
  # Test set performance
  tr = np.array(drfp_testset[1]).flatten() * 100
  pr = drfp_xgb.predict(drfp_testset[0]).flatten() * 100
  plt.scatter(pr, tr, alpha=0.7, marker=".")
  plt.xlabel("Predicted Yield", fontsize=10)
  plt.ylabel("Observed Yield", fontsize=10)
  x = np.linspace(0, 100, 100)
  y = np.linspace(0, 100, 100)
  plt.plot(x, y, linestyle="--", color="r")
  plt.title("Test set performance", fontsize=15)

  # Supporting Vector Machine
  from sklearn import svm
  from sklearn.svm import SVR

  rxnfp_svm = SVR(kernel='rbf', C=1.25)
  drfp_svm = SVR(kernel='rbf', C=1.25)
  rxnfp_svm.fit(rxnfp_trainset[0], rxnfp_trainset[1])
  drfp_svm.fit(drfp_trainset[0], drfp_trainset[1])

  print("Finish Training")
  print("Training Time: %.2f s" % (time.time() - start))  # 截止时间

  # Eval
  # trainset
  # R2
  rxnfp_train_R2 = R2(rxnfp_svm.predict(rxnfp_trainset[0]), np.array(rxnfp_trainset[1]))
  drfp_train_R2 = R2(drfp_svm.predict(drfp_trainset[0]), np.array(drfp_trainset[1]))
  # RMSE
  rxnfp_train_RMSE = RMSE(rxnfp_svm.predict(rxnfp_trainset[0]), np.array(rxnfp_trainset[1]))
  drfp_train_RMSE = RMSE(drfp_svm.predict(drfp_trainset[0]), np.array(drfp_trainset[1]))
  # MAE
  rxnfp_train_MAE = MAE(rxnfp_svm.predict(rxnfp_trainset[0]), np.array(rxnfp_trainset[1]))
  drfp_train_MAE = MAE(drfp_svm.predict(drfp_trainset[0]), np.array(drfp_trainset[1]))

  # R2
  rxnfp_test_R2 = R2(rxnfp_svm.predict(rxnfp_testset[0]), np.array(rxnfp_testset[1]))
  drfp_test_R2 = R2(drfp_svm.predict(drfp_testset[0]), np.array(drfp_testset[1]))
  # RMSE
  rxnfp_test_RMSE = RMSE(rxnfp_svm.predict(rxnfp_testset[0]), np.array(rxnfp_testset[1]))
  drfp_test_RMSE = RMSE(drfp_svm.predict(drfp_testset[0]), np.array(drfp_testset[1]))
  # MAE
  rxnfp_test_MAE = MAE(rxnfp_svm.predict(rxnfp_testset[0]), np.array(rxnfp_testset[1]))
  drfp_test_MAE = MAE(drfp_svm.predict(drfp_testset[0]), np.array(drfp_testset[1]))

  # Record
  f.write("Supporting Vector Machine:\n")

  f.write("rxnfp:\n")
  f.write("rxnfp_train_R2=%s\n" % rxnfp_train_R2)
  f.write("rxnfp_train_RMSE=%s\n" % rxnfp_train_RMSE)
  f.write("rxnfp_train_MAE=%s\n" % rxnfp_train_MAE)
  f.write("rxnfp_test_R2=%s\n" % rxnfp_test_R2)
  f.write("rxnfp_test_RMSE=%s\n" % rxnfp_test_RMSE)
  f.write("rxnfp_test_MAE=%s\n" % rxnfp_test_MAE)
  f.write("drfp:\n")
  f.write("drfp_train_R2=%s\n" % drfp_train_R2)
  f.write("drfp_train_RMSE=%s\n" % drfp_train_RMSE)
  f.write("drfp_train_MAE=%s\n" % drfp_train_MAE)
  f.write("drfp_test_R2=%s\n" % drfp_test_R2)
  f.write("drfp_test_RMSE=%s\n" % drfp_test_RMSE)
  f.write("drfp_test_MAE=%s\n" % drfp_test_MAE)
  f.write("\n")

  eval_metrics.loc["SVM-RXNFP-%s" % rs]["train_R2"] = rxnfp_train_R2
  eval_metrics.loc["SVM-RXNFP-%s" % rs]["train_RMSE"] = rxnfp_train_RMSE
  eval_metrics.loc["SVM-RXNFP-%s" % rs]["train_MAE"] = rxnfp_train_MAE
  eval_metrics.loc["SVM-RXNFP-%s" % rs]["test_R2"] = rxnfp_test_R2
  eval_metrics.loc["SVM-RXNFP-%s" % rs]["test_RMSE"] = rxnfp_test_RMSE
  eval_metrics.loc["SVM-RXNFP-%s" % rs]["test_MAE"] = rxnfp_test_MAE
  eval_metrics.loc["SVM-DRFP-%s" % rs]["train_R2"] = drfp_train_R2
  eval_metrics.loc["SVM-DRFP-%s" % rs]["train_RMSE"] = drfp_train_RMSE
  eval_metrics.loc["SVM-DRFP-%s" % rs]["train_MAE"] = drfp_train_MAE
  eval_metrics.loc["SVM-DRFP-%s" % rs]["test_R2"] = drfp_test_R2
  eval_metrics.loc["SVM-DRFP-%s" % rs]["test_RMSE"] = drfp_test_RMSE
  eval_metrics.loc["SVM-DRFP-%s" % rs]["test_MAE"] = drfp_test_MAE

  # SVM Figure
  # RXNFP
  fig = plt.figure(dpi=300, figsize=(10, 5))
  # Test set performance
  tr = np.array(rxnfp_testset[1]).flatten() * 100
  pr = rxnfp_svm.predict(rxnfp_testset[0]).flatten() * 100
  plt.scatter(pr, tr, alpha=0.7, marker=".")
  plt.xlabel("Predicted Yield", fontsize=10)
  plt.ylabel("Observed Yield", fontsize=10)
  x = np.linspace(0, 100, 100)
  y = np.linspace(0, 100, 100)
  plt.plot(x, y, linestyle="--", color="r")
  plt.title("Test set performance", fontsize=15)

  # DRFP
  # Figure
  fig = plt.figure(dpi=300, figsize=(10, 5))
  # Test set performance
  tr = np.array(drfp_testset[1]).flatten() * 100
  pr = drfp_svm.predict(drfp_testset[0]).flatten() * 100
  plt.scatter(pr, tr, alpha=0.7, marker=".")
  plt.xlabel("Predicted Yield", fontsize=10)
  plt.ylabel("Observed Yield", fontsize=10)
  x = np.linspace(0, 100, 100)
  y = np.linspace(0, 100, 100)
  plt.plot(x, y, linestyle="--", color="r")
  plt.title("Test set performance", fontsize=15)

  # kNN
  from sklearn.neighbors import KNeighborsRegressor

  # Train
  print("KNeighborsRegressor Start Training")
  start = time.time()

  rxnfp_knn = KNeighborsRegressor(n_neighbors=50)
  drfp_knn = KNeighborsRegressor(n_neighbors=50)
  rxnfp_knn.fit(rxnfp_trainset[0], rxnfp_trainset[1])
  drfp_knn.fit(drfp_trainset[0], drfp_trainset[1])

  print("Finish Training")
  print("Training Time: %.2f s" % (time.time() - start))  # 截止时间

  # Eval
  # trainset
  # R2
  rxnfp_train_R2 = R2(rxnfp_knn.predict(rxnfp_trainset[0]), np.array(rxnfp_trainset[1]))
  drfp_train_R2 = R2(drfp_knn.predict(drfp_trainset[0]), np.array(drfp_trainset[1]))
  # RMSE
  rxnfp_train_RMSE = RMSE(rxnfp_knn.predict(rxnfp_trainset[0]), np.array(rxnfp_trainset[1]))
  drfp_train_RMSE = RMSE(drfp_knn.predict(drfp_trainset[0]), np.array(drfp_trainset[1]))
  # MAE
  rxnfp_train_MAE = MAE(rxnfp_knn.predict(rxnfp_trainset[0]), np.array(rxnfp_trainset[1]))
  drfp_train_MAE = MAE(drfp_knn.predict(drfp_trainset[0]), np.array(drfp_trainset[1]))

  # R2
  rxnfp_test_R2 = R2(rxnfp_knn.predict(rxnfp_testset[0]), np.array(rxnfp_testset[1]))
  drfp_test_R2 = R2(drfp_knn.predict(drfp_testset[0]), np.array(drfp_testset[1]))
  # RMSE
  rxnfp_test_RMSE = RMSE(rxnfp_knn.predict(rxnfp_testset[0]), np.array(rxnfp_testset[1]))
  drfp_test_RMSE = RMSE(drfp_knn.predict(drfp_testset[0]), np.array(drfp_testset[1]))
  # MAE
  rxnfp_test_MAE = MAE(rxnfp_knn.predict(rxnfp_testset[0]), np.array(rxnfp_testset[1]))
  drfp_test_MAE = MAE(drfp_knn.predict(drfp_testset[0]), np.array(drfp_testset[1]))

  # Record
  f.write("KNeighbors Regressor:\n")
  f.write("params:\n")
  f.write("rxnfp KNN:%s\n" % rxnfp_knn.n_neighbors)
  f.write("drfp KNN:%s\n" % drfp_knn.n_neighbors)

  f.write("rxnfp:\n")
  f.write("rxnfp_train_R2=%s\n" % rxnfp_train_R2)
  f.write("rxnfp_train_RMSE=%s\n" % rxnfp_train_RMSE)
  f.write("rxnfp_train_MAE=%s\n" % rxnfp_train_MAE)
  f.write("rxnfp_test_R2=%s\n" % rxnfp_test_R2)
  f.write("rxnfp_test_RMSE=%s\n" % rxnfp_test_RMSE)
  f.write("rxnfp_test_MAE=%s\n" % rxnfp_test_MAE)
  f.write("drfp:\n")
  f.write("drfp_train_R2=%s\n" % drfp_train_R2)
  f.write("drfp_train_RMSE=%s\n" % drfp_train_RMSE)
  f.write("drfp_train_MAE=%s\n" % drfp_train_MAE)
  f.write("drfp_test_R2=%s\n" % drfp_test_R2)
  f.write("drfp_test_RMSE=%s\n" % drfp_test_RMSE)
  f.write("drfp_test_MAE=%s\n" % drfp_test_MAE)
  f.write("\n")

  eval_metrics.loc["kNN-RXNFP-%s" % rs]["train_R2"] = rxnfp_train_R2
  eval_metrics.loc["kNN-RXNFP-%s" % rs]["train_RMSE"] = rxnfp_train_RMSE
  eval_metrics.loc["kNN-RXNFP-%s" % rs]["train_MAE"] = rxnfp_train_MAE
  eval_metrics.loc["kNN-RXNFP-%s" % rs]["test_R2"] = rxnfp_test_R2
  eval_metrics.loc["kNN-RXNFP-%s" % rs]["test_RMSE"] = rxnfp_test_RMSE
  eval_metrics.loc["kNN-RXNFP-%s" % rs]["test_MAE"] = rxnfp_test_MAE
  eval_metrics.loc["kNN-DRFP-%s" % rs]["train_R2"] = drfp_train_R2
  eval_metrics.loc["kNN-DRFP-%s" % rs]["train_RMSE"] = drfp_train_RMSE
  eval_metrics.loc["kNN-DRFP-%s" % rs]["train_MAE"] = drfp_train_MAE
  eval_metrics.loc["kNN-DRFP-%s" % rs]["test_R2"] = drfp_test_R2
  eval_metrics.loc["kNN-DRFP-%s" % rs]["test_RMSE"] = drfp_test_RMSE
  eval_metrics.loc["kNN-DRFP-%s" % rs]["test_MAE"] = drfp_test_MAE

  # KNN Figure
  # RXNFP
  fig = plt.figure(dpi=300, figsize=(10, 5))
  # Test set performance
  tr = np.array(rxnfp_testset[1]).flatten() * 100
  pr = rxnfp_knn.predict(rxnfp_testset[0]).flatten() * 100
  plt.scatter(pr, tr, alpha=0.7, marker=".")
  plt.xlabel("Predicted Yield", fontsize=10)
  plt.ylabel("Observed Yield", fontsize=10)
  x = np.linspace(0, 100, 100)
  y = np.linspace(0, 100, 100)
  plt.plot(x, y, linestyle="--", color="r")
  plt.title("Test set performance", fontsize=15)

  # DRFP
  # Figure
  fig = plt.figure(dpi=300, figsize=(10, 5))
  # Test set performance
  tr = np.array(drfp_testset[1]).flatten() * 100
  pr = drfp_knn.predict(drfp_testset[0]).flatten() * 100
  plt.scatter(pr, tr, alpha=0.7, marker=".")
  plt.xlabel("Predicted Yield", fontsize=10)
  plt.ylabel("Observed Yield", fontsize=10)
  x = np.linspace(0, 100, 100)
  y = np.linspace(0, 100, 100)
  plt.plot(x, y, linestyle="--", color="r")
  plt.title("Test set performance", fontsize=15)

  f.close()


In [ ]:
# Evaluation metrics report
for i in range(len(rs_list), eval_metrics.shape[0], len(rs_list)+1):
  for j in range(eval_metrics.shape[1]):
    eval_metrics.iloc[i,j] = "%.4f ± %.4f" % (eval_metrics.iloc[i-len(rs_list):i-1,j].mean(), eval_metrics.iloc[i-len(rs_list):i-1,j].std())
eval_metrics.to_csv("%s/exp/Heck_JCP/ML_report_%s.csv" % (doc_name, datetime.datetime.now()))
print(eval_metrics)